# Markout analysis

**Sign conventions are stated per section and named in the code** (`client_signed` /
`firm_signed`) — the double negative "client-negative means the firm gains" is the classic
trap in markout work.

| section | convention | question it answers |
|---|---|---|
| **A** | client-signed | Is the flow toxic, and does our selection rule keep the bad half? |
| **B** | client-signed | Which clients are toxic? (per-client ranking) |
| **C** | **firm-signed** | What is an internalized fill actually worth to the book? |
| **D** | — | Which horizon matches the book's realised holding time? |

For every fill the **markout at horizon h** is the signed mid move after the fill,
$m_h = s\cdot(mid_{t+h} - mid_t)$, with $s=+1$ for a client BUY under the client
convention and the opposite sign under the firm convention (the firm takes the other side
of every principal fill).

Fills whose horizon runs past the last quote are **dropped at that horizon** rather than
clamped to the close, so no end-of-day bias creeps in. One engineered day: treat every
ranking as illustrative, not statistically significant.

In [ ]:
import sys
from bisect import bisect_right          # binary search over the quote tape
from datetime import datetime, timedelta

import pandas as pd

sys.path.insert(0, '.')                  # import the engine package from the repo root
import config                            # all paths and risk params live here
from internalizer.data import load_quotes

quotes = load_quotes(config.QUOTES_CSV)  # tape as Quote objects, prices in cents
qts = [q.ts for q in quotes]             # timestamps, ascending -> bisect key
mids = [(q.bid + q.ask) / 2 for q in quotes]  # fair value at each tick
LAST = qts[-1]                           # horizons past this are dropped, never clamped

def mid_at(ts):
    """Prevailing mid (cents) at or before ts."""
    return mids[bisect_right(qts, ts) - 1]   # -1 = last quote at or before ts

fills = pd.read_csv(config.FILLS_CSV, parse_dates=['timestamp'])  # any version's output
print(f'{len(fills)} fills, {fills.client_id.nunique()} clients')
fills.head(3)

---
## Section A — client-direction markout across ALL client flow

**Convention: client-signed throughout this section.** `+1` for a client BUY, `-1` for a
SELL, so **positive = the market moved the client's way after they traded = toxic flow**,
and the firm, holding the other side, loses. (Section C below switches to the firm's own
direction to price the inventory; each section states its convention in the heading.)

Every internalized fill is flow we *chose* to take, so markout on internalized fills alone
measures **flow quality × our selection rule**, not flow quality. To separate the two, this
section marks out every venue the flow could land in, plus the orders that never traded:

- **INTERNAL** — we took the risk.
- **CROSS** — matched against another client; we took no risk.
- **MARKET** — routed out; the risk went to the street.
- **NOT FILLED** — cancelled IOC / expired DAY, marked out from **arrival** (there is no
  execution to mark from). This is the flow we declined, and it is the natural control group.

If INTERNAL markouts were consistently worse than MARKET, the acceptance rule would be
adversely selecting toxic flow into the book.

In [ ]:
from internalizer.data import load_orders

ALL_HORIZONS = [('30s', 30), ('1min', 60), ('5min', 300), ('30min', 1800)]

orders = load_orders(config.ORDERS_CSV)                  # includes orders that never traded
filled_qty = fills.groupby('order_id')['quantity'].sum()  # executed shares per order


def client_signed_markout(ts, side, qty, horizons=ALL_HORIZONS):
    """Client-signed mid move: +1 for BUY, -1 for SELL. Positive = toxic for us."""
    client_sign = 1 if side == 'BUY' else -1   # the convention, named not commented
    m0 = mid_at(ts)                            # fair value at the fill (or arrival)
    row = {}
    for name, secs in horizons:
        t1 = ts + timedelta(seconds=secs)
        # drop horizons that run past the tape rather than clamping to the close
        row[name] = client_sign * (mid_at(t1) - m0) * qty if t1 <= LAST else None
    return row


records = []
for _, f in fills.iterrows():                      # every execution, all three venues
    r = {'venue': f['venue'], 'qty': f['quantity'],
         'client': f['client_id'].replace('CLIENT_', '')}
    r.update(client_signed_markout(f['timestamp'], f['side'], f['quantity']))
    records.append(r)

for o in orders:                                   # the flow we declined, marked from arrival
    leaves = o.quantity - int(filled_qty.get(o.order_id, 0))   # unexecuted shares
    if leaves <= 0:
        continue                                   # fully filled -> already counted above
    r = {'venue': 'NOT FILLED', 'qty': leaves,
         'client': o.client_id.replace('CLIENT_', '')}
    r.update(client_signed_markout(o.ts, o.side, leaves))      # arrival, not execution
    records.append(r)

flow = pd.DataFrame(records)


def weighted(g, horizons=ALL_HORIZONS):
    """Share-weighted markout in cents/share (values are already cents x shares)."""
    out = {'shares': int(g['qty'].sum())}
    for name, _ in horizons:
        v = g.dropna(subset=[name])                # only fills that reached this horizon
        out[f'{name} (c/sh)'] = round(v[name].sum() / v['qty'].sum(), 2) if len(v) else None
    return pd.Series(out)


by_venue = flow.groupby('venue').apply(weighted, include_groups=False)
by_venue.loc['— ALL FLOW —'] = weighted(flow)      # whole-day flow character
by_venue   # client-signed: positive = flow was right, i.e. toxic to whoever took it

In [ ]:
# selection check: INTERNAL vs MARKET vs the declined flow, side by side
sel = by_venue.loc[[v for v in ['INTERNAL', 'MARKET', 'CROSS', 'NOT FILLED'] if v in by_venue.index]]
gap = (sel.loc['INTERNAL'] - sel.loc['MARKET']).drop('shares')   # what our rule kept vs routed
print('INTERNAL minus MARKET (client-signed c/sh; negative = we kept the more benign flow):')
print(gap.to_string(), '\n')
sel

**Reading Section A (client-signed: positive = flow was right = toxic to its counterparty):**

| venue | shares | 30s | 1min | 5min | 30min |
|---|---|---|---|---|---|
| INTERNAL | 228,300 | −0.16 | −1.37 | −5.03 | −10.22 |
| MARKET | 248,400 | −1.18 | −1.45 | −2.42 | −5.84 |
| CROSS | 1,400 | 0.00 | 0.00 | 0.00 | 0.00 |
| NOT FILLED | 45,900 | +0.88 | +2.84 | +10.59 | +6.38 |
| ALL FLOW | 524,000 | −0.55 | −1.03 | −2.53 | −6.75 |

- **The whole day's client flow was benign** (−6.75¢ at 30 minutes): prices moved against
  clients after they traded, which is why the firm's inventory drift was positive. The
  selection rule is not creating that — it is a property of the flow.
- **Selection check**: INTERNAL minus MARKET is +1.02¢ at 30s but −2.61¢ at 5min and
  −4.38¢ at 30min. On the horizon that matches the book's realised holding time (~5 min,
  Section D), the flow we kept was **more benign than the flow we routed** — the acceptance
  rule is not adversely selecting. The 30-second sign flip is worth remembering rather than
  explaining away: at the horizon of a fast hedge we keep the slightly worse half.

**Two mechanical artefacts, not findings:**

1. **CROSS is exactly 0.00 by construction.** Both legs print at the same time and price with
   opposite client signs, so their markouts cancel in any aggregate. Read crosses one leg at
   a time or not at all.
2. **NOT FILLED is biased positive and is not a clean control.** A resting buy limit goes
   unfilled *precisely because* the price rose away from it — non-fill and adverse-for-us
   price movement are the same event, so the +10.59¢ is largely mechanical, not evidence the
   declined flow was informed. A real control needs matched orders that could have filled
   but did not for reasons unrelated to price (queue position, size), which this data cannot
   provide.

---
## Section B — client-direction markout, per client

Same client-signed convention as Section A, grouped by client to rank flow toxicity.


In [ ]:
HORIZONS = {'30s': 30, '2min': 120, '10min': 600}   # Section B's own horizon set

rows = []
for _, f in fills.iterrows():
    s = 1 if f['side'] == 'BUY' else -1          # client-signed direction
    m0 = mid_at(f['timestamp'])                  # mid at fill time
    row = {'client': f['client_id'].replace('CLIENT_', ''), 'qty': f['quantity'],
           'venue': f['venue']}
    for name, secs in HORIZONS.items():
        t1 = f['timestamp'] + timedelta(seconds=secs)
        # drop fills whose horizon runs past the tape (avoids clamping bias)
        row[name] = s * (mid_at(t1) - m0) if t1 <= LAST else None   # per-share, not per-fill
    rows.append(row)
mo = pd.DataFrame(rows)

def share_weighted(g):
    out = {'fills': len(g), 'shares': g['qty'].sum()}
    for h in HORIZONS:
        v = g.dropna(subset=[h])                 # horizon-specific sample
        out[f'markout {h} (c/sh)'] = round((v[h] * v['qty']).sum() / v['qty'].sum(), 2)
    return pd.Series(out)

per_client = (mo.groupby('client').apply(share_weighted, include_groups=False)
                .sort_values('markout 10min (c/sh)', ascending=False))
per_client   # positive at the top = most toxic (market moves their way after they trade)

In [ ]:
# aggregate flow character + internalized-only view (what the firm actually absorbed)
total = share_weighted(mo)                        # every execution
internal = share_weighted(mo[mo.venue == 'INTERNAL'])   # the subset we chose to take
pd.DataFrame({'all fills': total, 'internalized only': internal})

In [ ]:
# Can the per-client ranking above support a toxicity conclusion? Three diagnostics.
import random
from collections import defaultdict

random.seed(11)                                   # reproducible bootstrap draws
CRASH = (pd.Timestamp('2026-08-17 10:55'), pd.Timestamp('2026-08-17 11:30'))  # the selloff
SECS = 600                                        # the horizon Section B ranks on


def client_rows(exclude_crash=False):
    """(markout, qty, ts, 30-min block) per fill, client-signed, at SECS."""
    per = defaultdict(list)
    for _, f in fills.iterrows():
        t1 = f['timestamp'] + pd.Timedelta(seconds=SECS)
        if t1 > LAST or (exclude_crash and CRASH[0] <= f['timestamp'] <= CRASH[1]):
            continue                              # off-tape, or the event under test
        q = f['quantity']
        client_signed = q if f['side'] == 'BUY' else -q
        m0 = (round(f['nbbo_bid'] * 100) + round(f['nbbo_ask'] * 100)) / 2   # mid from the CSV
        block = f['timestamp'].hour * 2 + (f['timestamp'].minute >= 30)      # 30-min block id
        per[f['client_id'].replace('CLIENT_', '')].append(
            (client_signed * (mid_at(t1) - m0), q, f['timestamp'], block))
    return per


rows = []
for c, rs in client_rows().items():
    tot = sum(p for p, _, _, _ in rs); sh = sum(q for _, q, _, _ in rs)
    absmo = sum(abs(p) for p, _, _, _ in rs)      # denominator for concentration measures
    blocks = defaultdict(list)
    for p, q, _, b in rs:
        blocks[b].append((p, q))                  # group fills by 30-min block
    keys = list(blocks)
    sims = []
    for _ in range(3000):                         # block bootstrap: fills in a block are correlated
        pick = [blocks[random.choice(keys)] for _ in keys]   # resample blocks with replacement
        pp = sum(x for b in pick for x, _ in b); ss = sum(q for b in pick for _, q in b)
        if ss:
            sims.append(pp / ss)
    sims.sort()
    lo, hi = sims[int(.025 * len(sims))], sims[int(.975 * len(sims))]   # 95% percentile CI
    rows.append({'client': c, 'fills': len(rs), 'shares': sh,
                 'markout 10min': round(tot / sh, 2),
                 # share of the client's volume where the market moved their way
                 'hit rate %': round(sum(q for p, q, _, _ in rs if p > 0) / sh * 100),
                 # how much of the result rides on a handful of fills
                 'top-3 fills % of |markout|': round(sum(sorted((abs(p) for p, _, _, _ in rs),
                                                                reverse=True)[:3]) / absmo * 100),
                 # how much rides on one 35-minute market event
                 'crash-window % of |markout|': round(sum(abs(p) for p, _, ts, _ in rs
                                                          if CRASH[0] <= ts <= CRASH[1]) / absmo * 100),
                 'CI low': round(lo, 2), 'CI high': round(hi, 2),
                 'excludes 0': 'YES' if (lo > 0 or hi < 0) else 'no'})
diagnostics = pd.DataFrame(rows).set_index('client').sort_values('markout 10min', ascending=False)
diagnostics

In [ ]:
# Rank stability: does the ranking survive removing one 35-minute market event?
def ranking(exclude_crash):
    per = client_rows(exclude_crash)
    return {c: sum(p for p, _, _, _ in rs) / sum(q for _, q, _, _ in rs)   # share-weighted
            for c, rs in per.items()}


full, ex = ranking(False), ranking(True)          # same clients, one window removed
order_full = sorted(full, key=full.get, reverse=True)   # most toxic first
order_ex = sorted(ex, key=ex.get, reverse=True)
stability = pd.DataFrame([
    {'client': c, 'full day': round(full[c], 2), 'rank': order_full.index(c) + 1,
     'ex-crash': round(ex[c], 2), 'rank (ex)': order_ex.index(c) + 1,
     'rank move': order_ex.index(c) - order_full.index(c)}
    for c in order_full]).set_index('client')

d2 = sum((order_full.index(c) - order_ex.index(c)) ** 2 for c in full)   # sum of squared rank gaps
n = len(full)
print(f'Spearman rank correlation, full-day vs ex-crash: {1 - 6 * d2 / (n * (n*n - 1)):.2f}')
stability

**Can Section B support a client-toxicity conclusion? No — and the diagnostics say why.**

The ranking looks decisive (G at +16.8¢, D at −18.1¢) but does not survive scrutiny:

1. **Significance**: one client of ten has a bootstrap CI excluding zero (I, upper bound
   −0.67). Ten tests at α = 0.05 produce 0.5 false positives in expectation, so one hit *is*
   the null result; nothing survives a multiplicity correction.
2. **Concentration**: 34–71% of each client's absolute markout comes from their **top three
   fills**. Effective sample size is single digits, not the 13–95 fills shown.
3. **Event dependence**: the extreme clients (G, B, H, D) draw 55–61% of their markout from
   the 10:55–11:30 selloff.
4. **Rank stability**: removing that single 35-minute window moves G from rank 1 to rank 9
   and D from rank 10 to rank 3. **Spearman correlation between the two rankings is 0.07** —
   indistinguishable from a reshuffle. The spread also collapses from [−18, +17] to
   [−4.3, +4.3], so ~87% of the cross-sectional dispersion is that one event.

**The cause is structural, not sample size.** Clients trading in the same window are marked
against the *same* price path, so their markouts are mechanically correlated. "G is toxic,
D is benign" mostly restates "during the 11:00 move G was on the up side and D on the down
side" — the ranking measures **when they traded and which side**, not client-specific
information. Adding days does not fix this on its own; the fix is stratifying by market
regime and requiring significance **within** each stratum, which is why per-client markout
tiering sits in the production list as a data-collection project rather than a conclusion
this day can deliver.

**What the table *is* good for**: as a monitoring dashboard once weeks of data exist, and as
the shape of the argument — the columns (hit rate, concentration, event share, CI, rank
stability) are the checks any real tiering proposal has to pass before it changes pricing.

---
## Section C — markout-adjusted edge (**firm-signed**)

Everything above is **client-signed** (positive = toxic flow). The firm holds the
opposite side, so **firm P&L = −(client-signed markout)**. This double negative is the
classic sign-error trap in markout work, so from here on both conventions are named
explicitly in the code (`client_signed` / `firm_signed`) rather than left to a comment.

For each internalized fill the firm's economics have two parts:

$$\text{adjusted edge}_h \;=\; \underbrace{(mid_t - px)\cdot q_{firm}}_{\text{execution edge}} \;+\; \underbrace{q_{firm}\cdot(mid_{t+h} - mid_t)}_{\text{firm markout}}$$

- **execution edge**: what the fill earned against fair value at that instant (0 on even
  spreads where mid is a whole cent, +0.5¢ on odd spreads where rounding favours the firm).
- **firm markout**: what the inventory taken on would have been worth h later, if held.

Reads only `fills.csv` + the quote tape, so any version's output can be analysed
without re-running the engine: point `FILLS_PATH` at `out/fills.csv`, a saved
`out/fills_v0.csv`, etc. `nbbo_bid`/`nbbo_ask` are recorded on every fill row, so the
spread bucket and the fair value at fill time both come straight from the CSV.

**Caveat on horizon choice**: markout assumes each fill's full size is held in isolation
for h. The book actually nets 228,300 gross internalized shares down to ~1,559 shares of
time-weighted exposure (146× turnover), so long horizons overstate what the firm really
carried. Match the horizon to realised holding time; the 1-minute column is the honest
one here.

In [ ]:
FILLS_PATH = config.FILLS_CSV        # any version's fills.csv works; no engine replay
ADJ_HORIZONS = [('5s', 5), ('30s', 30), ('1min', 60), ('5min', 300), ('30min', 1800)]


def markout_adjusted_edge(fills_path=FILLS_PATH, horizons=ADJ_HORIZONS):
    """Per-spread-bucket execution edge and firm markout, straight from a fills CSV.

    Sign convention, spelled out once: `client_signed` is +qty for a client BUY;
    the firm takes the other side, so `firm_signed = -client_signed`. Firm markout
    is firm_signed x (mid_future - mid_fill): a short book gains when the mid falls.
    """
    df = pd.read_csv(fills_path, parse_dates=['timestamp'])
    df = df[df['venue'] == 'INTERNAL'].copy()          # principal risk only

    px = (df['price'] * 100).round()                    # cents
    bid = (df['nbbo_bid'] * 100).round()                # NBBO recorded on the fill row
    ask = (df['nbbo_ask'] * 100).round()
    df['spread_c'] = (ask - bid).astype(int)            # bucket key: edge depends on parity
    df['mid'] = (bid + ask) / 2                         # fair value at fill time
    df['client_signed'] = df['quantity'].where(df['side'] == 'BUY', -df['quantity'])
    df['firm_signed'] = -df['client_signed']            # firm always takes the other side
    df['edge'] = (df['mid'] - px) * df['firm_signed']   # cents x shares, firm view

    for name, secs in horizons:                         # firm markout at each horizon
        t1 = df['timestamp'] + pd.Timedelta(seconds=secs)
        future_mid = [mid_at(t) if t <= LAST else None for t in t1]   # None = off-tape
        df[f'mo_{name}'] = [None if m is None else fs * (m - m0)
                            for m, fs, m0 in zip(future_mid, df['firm_signed'], df['mid'])]

    rows = {}
    for spr, g in df.groupby('spread_c'):
        r = {'shares': int(g['quantity'].sum()),
             'edge (c/sh)': round(g['edge'].sum() / g['quantity'].sum(), 2)}
        for name, _ in horizons:                        # drop fills whose horizon runs off the tape
            v = g.dropna(subset=[f'mo_{name}'])
            r[f'adj {name}'] = (round((v['edge'].sum() + v[f'mo_{name}'].sum())
                                      / v['quantity'].sum(), 2) if len(v) else None)
        rows[f'{spr}c'] = r
    out = pd.DataFrame(rows).T

    all_r = {'shares': int(df['quantity'].sum()),      # same computation, ungrouped
             'edge (c/sh)': round(df['edge'].sum() / df['quantity'].sum(), 2)}
    for name, _ in horizons:
        v = df.dropna(subset=[f'mo_{name}'])
        all_r[f'adj {name}'] = round((v['edge'].sum() + v[f'mo_{name}'].sum())
                                     / v['quantity'].sum(), 2)
    out.loc['ALL'] = all_r
    return out, df


adjusted, internal_fills = markout_adjusted_edge()
adjusted   # cents/share, firm view: execution edge + inventory markout at each horizon

In [ ]:
# dollar view: how little of the value is execution edge once inventory is marked
dollars = []
for name, _ in ADJ_HORIZONS:
    v = internal_fills.dropna(subset=[f'mo_{name}'])   # horizon-specific sample
    dollars.append({'horizon': name,
                    'shares': int(v['quantity'].sum()),
                    'edge ($)': round(v['edge'].sum() / 100, 0),          # cents -> dollars
                    'firm markout ($)': round(v[f'mo_{name}'].sum() / 100, 0),
                    'adjusted ($)': round((v['edge'].sum() + v[f'mo_{name}'].sum()) / 100, 0)})
pd.DataFrame(dollars).set_index('horizon')

**Reading the adjusted table (firm view, cents/share):**

- **Wider spreads pay better once risk-adjusted.** The 6¢ bucket reaches +212.5¢/share at
  30 minutes; the 1¢ bucket is the only one that turns negative (−15.35¢). On this day wide
  spreads were compensation, not a warning — the flow arriving at wide spreads was benign.
  Note this is the empirical half of the "wide spreads cut both ways" argument; the
  analytical half (the exit costs half that same wide spread) still stands.
- **Execution edge is a rounding error against inventory P&L.** At 30 minutes, $400 of the
  $21,051 is execution edge — 2%. All the discussion of half-cent tick rounding lives inside
  that 2%.
- **The 1¢ bucket inverts with horizon**: highest edge (0.5¢, a 100% capture of the
  half-spread) but negative long-horizon adjusted P&L, because those are reduce-branch fills
  that close out positions which were still appreciating. Bleeding inventory away therefore
  costs edge but protects markout — the two metrics disagree by construction.
- **Horizon is a policy choice, not a detail**: the same fills read +0.22¢ at 5 seconds and
  +10.42¢ at 30 minutes, a 47× spread. Choose the horizon that matches realised holding time.

---
## Section D — which horizon matches the book? Realised holding time

The adjusted table only means something at the horizon the firm actually carries
inventory for. Two independent estimates, both from the output CSVs:

1. **FIFO share-level pairing** — every closing share is matched against the oldest open
   share, giving a share-weighted average holding time plus its full distribution.
2. **Little's law** — average inventory ÷ one-way flow rate. In steady state this equals
   the mean residence time, so agreement between the two is evidence the day was close to
   steady state rather than one long directional build.

A high turnover ratio is *not* evidence of short holding: a small book that is constantly
replenished still holds each individual share for a while. Turnover measures flow against
inventory; holding time measures residence.

In [ ]:
from collections import deque


def principal_trade_stream(fills_path=FILLS_PATH, firm_path=config.FIRM_TRADES_CSV):
    """(timestamp, firm_signed_qty, fill_row_or_None) for every trade carrying firm risk."""
    ev = []
    f = pd.read_csv(fills_path, parse_dates=['timestamp'])
    for _, r in f[f['capacity'] == 'PRINCIPAL'].iterrows():   # cross/route carry no firm risk
        firm_signed = -r['quantity'] if r['side'] == 'BUY' else r['quantity']
        ev.append((r['timestamp'], firm_signed, r))
    h = pd.read_csv(firm_path, parse_dates=['timestamp'])     # the firm's own hedges
    for _, r in h.iterrows():
        firm_signed = r['quantity'] if r['side'] == 'BUY' else -r['quantity']
        ev.append((r['timestamp'], firm_signed, None))        # no client row for a hedge
    ev.sort(key=lambda x: x[0])                               # chronological
    return ev


def holding_time(ev):
    """FIFO share-weighted holding time + Little's law cross-check + percentiles."""
    lots, durations = deque(), []                       # open lots: (ts, signed qty)
    for ts, d, _ in ev:
        while d and lots and (lots[0][1] > 0) != (d > 0):   # opposite sign closes the lot
            t0, q0 = lots[0]
            m = min(abs(q0), abs(d))                    # shares this pairing closes
            durations.append(((ts - t0).total_seconds(), m))
            q0 -= m if q0 > 0 else -m                   # shrink the open lot
            d -= -m if d < 0 else m                     # shrink the incoming trade
            lots[0] = (t0, q0)
            if q0 == 0:
                lots.popleft()                          # lot fully closed
        if d:
            lots.append((ts, d))                        # residual opens a new lot

    closed = sum(m for _, m in durations)
    fifo_mean = sum(s * m for s, m in durations) / closed    # share-weighted mean residence

    span = (ev[-1][0] - ev[0][0]).total_seconds()       # Little's law inputs
    pos = gross = 0
    tw = 0.0
    prev = None
    for ts, d, _ in ev:
        if prev is not None:
            tw += abs(pos) * (ts - prev).total_seconds()    # integral of |position| dt
        pos += d
        gross += abs(d)                                 # both directions
        prev = ts
    avg_inventory = tw / span
    one_way_flow = gross / 2 / span                     # shares per second, one side
    littles = avg_inventory / one_way_flow              # = mean residence in steady state

    durations.sort()                                     # share-weighted percentiles
    cum, pct = 0, {}
    for sec, m in durations:
        cum += m
        for p in (25, 50, 75, 90):
            pct.setdefault(p, sec) if cum >= closed * p / 100 else None
    return {'FIFO mean (s)': round(fifo_mean),
            "Little's law (s)": round(littles),
            'p25 (s)': pct[25], 'p50 (s)': pct[50], 'p75 (s)': pct[75], 'p90 (s)': pct[90],
            'avg inventory (sh)': round(avg_inventory),
            'gross traded (sh)': gross}


ev = principal_trade_stream()
pd.Series(holding_time(ev))   # FIFO and Little's law agree -> steady-state turnover

In [ ]:
# Is the 1c bucket entirely risk-reducing? (it should be: at 1c only the reduce
# branch can fill, capped at min(order.remaining, abs(position)) and priced at the touch)
pos, cls = 0, []
for ts, firm_signed, row in ev:
    if row is not None and row['venue'] == 'INTERNAL':
        spread_c = round((row['nbbo_ask'] - row['nbbo_bid']) * 100)
        if spread_c == 1:
            at_touch = (round(row['price'] * 100) ==      # buy at ask / sell at bid
                        round((row['nbbo_ask'] if row['side'] == 'BUY' else row['nbbo_bid']) * 100))
            effect = ('flat-start' if pos == 0 else      # no inventory to reduce
                      'reduce' if (pos > 0) != (firm_signed > 0) else 'increase')
            cls.append({'effect': effect, 'at_touch': at_touch, 'qty': row['quantity'],
                        'flips_sign': abs(pos) < row['quantity']})   # would overshoot through zero
    pos += firm_signed                                   # advance the position path

c = pd.DataFrame(cls)
print(f"1c INTERNAL fills: {len(c)} fills / {c['qty'].sum():,} shares")
print(f"  all at the touch : {c['at_touch'].all()}")
print(f"  any sign flip    : {c['flips_sign'].any()}")   # the min(..., abs(position)) cap
c.groupby('effect')['qty'].agg(fills='count', shares='sum')